In [1]:
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("AI-MO/NuminaMath-1.5")

In [2]:
# import sys
# from pathlib import Path

# # 1. See where the notebook thinks it is
# print("CWD:", Path().resolve())

# # 2. See what's actually in the parent
# print("\nParent contents:")
# for p in Path().resolve().parent.iterdir():
#     print(" ", p)

# # 3. Find config.py anywhere nearby
# print("\nSearching for config.py:")
# for p in Path().resolve().parents[2].rglob("config.py"):
#     print(" ", p)

In [3]:
ds["train"][0]

{'problem': '\nProblem 1. Find all prime numbers $p$ for which there exist positive integers $x, y$ and $z$ such that the number\n\n$$\nx^{p}+y^{p}+z^{p}-x-y-z\n$$\n\nis a product of exactly three distinct prime numbers.\n',
 'solution': "\nSolution. Let $A=x^{p}+y^{p}+z^{p}-x-y-z$. For $p=2$, we take $x=y=4$ and $z=3$. Then $A=30=2 \\cdot 3 \\cdot 5$. For $p=3$ we can take $x=3$ and $y=2$ and $z=1$. Then again $A=30=2 \\cdot 3 \\cdot 5$. For $p=5$ we can take $x=2$ and $y=1$ and $z=1$. Again $A=30=2 \\cdot 3 \\cdot 5$.\n\nAssume now that $p \\geqslant 7$. Working modulo 2 and modulo 3 we see that $A$ is divisible by both 2 and 3. Moreover, by Fermat's Little Theorem, we have\n\n$$\nx^{p}+y^{p}+z^{p}-x-y-z \\equiv x+y+z-x-y-z=0 \\bmod p \\text {. }\n$$\n\nTherefore, by the given condition, we have to solve the equation\n\n$$\nx^{p}+y^{p}+z^{p}-x-y-z=6 p\n$$\n\nIf one of the numbers $x, y$ and $z$ is bigger than or equal to 2 , let's say $x \\geqslant 2$, then\n\n$$\n6 p \\geqslant x^{p

In [4]:
import json
import os
import re
import sys
import glob
from pathlib import Path

import requests
from dotenv import load_dotenv

load_dotenv()

load_dotenv()

API_KEY = os.getenv("PCSS_API_KEY", "")
BASE_URL = os.getenv("PCSS_BASE_URL", "https://llm.hpc.psnc.pl/v1/chat/completions")
MODEL = os.getenv("PCSS_MODEL", "llama3.3:70b")

In [5]:

def call_llm(system_prompt: str, user_prompt: str) -> str:
    """
    Call the LLM API (OpenAI-compatible).
    Swap BASE_URL / auth headers here when moving to PCSS.
    """
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        "temperature": 0.2,
    }
    response = requests.post(
        BASE_URL,
        headers=headers,
        json=payload,
        timeout=600,
    )
    response.raise_for_status()
    return response.json()["choices"][0]["message"]["content"].strip()


In [6]:
TRANSLATE_SYSTEM_PROMPT = """You are an expert mathematical translator specializing in English to Polish translation for olympiad and academic mathematics.

STRICT RULES:
1. Translate ALL natural language text from English to Polish
2. Keep ALL LaTeX commands, formulas, and math expressions EXACTLY as they are — do not modify anything inside $...$ or $$...$$
3. Keep variable names, labels, and point names unchanged (e.g. $A$, $B$, $ABC$, $f(x)$)
4. Output ONLY the translated text — no explanations, no comments, no preamble
5. Preserve all formatting, newlines, and structure from the original
6. Use correct Polish grammatical forms — pay special attention to:
   - noun declension (e.g. "trójkąt" → "trójkąta", "trójkątowi", "trójkącie")
   - adjective agreement (e.g. "prostokątny" must agree in gender/case with its noun)
   - verb conjugation (e.g. "udowodnij", "wyznacz", "oblicz", "pokaż, że")
   - preposition + case agreement (e.g. "dla trójkąta", "w okręgu", "na prostej")
   - correct mathematical terminologuy translation (e.g. calculus -> analiza matematyczna, )

EXAMPLE:
Input:
"Problem 3. Triangle $ABC$ is such that $AB < AC$. The perpendicular bisector of side $BC$ intersects lines $AB$ and $AC$ at points $P$ and $Q$, respectively. Let $H$ be the orthocentre of triangle $ABC$, and let $M$ and $N$ be the midpoints of segments $BC$ and $PQ$, respectively. Prove that lines $HM$ and $AN$ meet on the circumcircle of $ABC$."

Output:
"Zadanie 3. Trójkąt $ABC$ jest taki, że $AB < AC$. Symetralna boku $BC$ przecina proste $AB$ i $AC$ odpowiednio w punktach $P$ i $Q$. Niech $H$ będzie ortocentrum trójkąta $ABC$, a $M$ i $N$ — środkami odcinków $BC$ i $PQ$. Udowodnij, że proste $HM$ i $AN$ przecinają się na okręgu opisanym na trójkącie $ABC$."

COMMON POLISH MATH VOCABULARY:

"""

TRANSLATE_PROBLEM_PROMPT = """Translate the following math problem from English to Polish.
Output ONLY the Polish translation, nothing else.  

TEXT TO TRANSLATE:
{text}"""

TRANSLATE_SOLUTION_PROMPT = """Translate the following math solution from English to Polish.
Output ONLY the Polish translation, nothing else.
Preserve all mathematical steps, formulas, and logical structure.

TEXT TO TRANSLATE:
{text}"""

In [7]:

import json
from pathlib import Path
def save_translation(output_file: Path, idx: int, task: dict, problem_pl: str, solution_pl: str):
    record = {
        "id": idx,
        "problem_en": task["problem"],
        "problem_pl": problem_pl,
        "solution_en": task["solution"],
        "solution_pl": solution_pl,
    }
    with open(output_file, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

OUTPUT_FILE = Path("../data/translations_output.jsonl")  
save_translation

<function __main__.save_translation(output_file: pathlib.Path, idx: int, task: dict, problem_pl: str, solution_pl: str)>

In [8]:
ds["train"][1133]

{'problem': 'III OM - I - Problem 7\n\nLet $ a $, $ b $ denote the legs of a right triangle, $ c $ - its hypotenuse, $ r $ - the radius of the inscribed circle, and $ r_a $, $ r_b $, $ r_c $ - the radii of the excircles of this triangle. Prove that:\n1) $ r + r_a + r_b = r_c $\n2) the radii $ r $, $ r_a $, $ r_b $, $ r_c $ are simultaneously integers if and only if the sides $ a $, $ b $, $ c $ are expressed by integers.',
 'solution': 'In this task, we are dealing with figure 16, which is a special case of figure 11, as angle \\( C \\) is a right angle. We see that \\( r = CM \\), \\( r_a = CP \\), \\( r_b = CR \\), \\( r_c = CQ \\).\nBy denoting \\( a + b + c = 2p \\) and applying the formulas given in Note II to problem 16, we obtain\n\n\n\n\n\n\nFrom this, the first part of the theorem immediately follows:\n\n\n\n\n\n\nTo prove the second part, let us first assume that \\( a \\), \\( b \\), and \\( c \\) are integers. The proof that \\( r \\), \\( r_a \\), \\( r_b \\), and \\( r_c 

In [12]:
%%time
NUM = 1551
def get_processed_ids(output_file: Path) -> set:
    if not output_file.exists():
        return set()
    with open(output_file, "r", encoding="utf-8") as f:
        return {json.loads(line)["id"] for line in f if line.strip()}

processed = get_processed_ids(OUTPUT_FILE)
print(f"Already processed: {len(processed)} records")

for i in range(min(NUM, len(ds["train"]))):
    if i in processed:
        print(f"Skipping {i} (already done)")
        continue

    task = ds["train"][i]

    try:
        problem_pl = call_llm(TRANSLATE_SYSTEM_PROMPT, TRANSLATE_PROBLEM_PROMPT.format(text=task["problem"]))
        # print("=== PROBLEM ===", i)
        # print("[EN]", task['problem'])
        # print("[PL]", problem_pl)
        solution_pl = call_llm(TRANSLATE_SYSTEM_PROMPT, TRANSLATE_SOLUTION_PROMPT.format(text=task["solution"]))

        save_translation(OUTPUT_FILE, i, task, problem_pl, solution_pl)
        print(f"[{i+1}/{len(ds['train'])}] ✓ saved")

    except Exception as e:
        print(f"[{i}] ✗ Error: {e} — skipping")
        continue




Already processed: 1550 records
Skipping 0 (already done)
Skipping 1 (already done)
Skipping 2 (already done)
Skipping 3 (already done)
Skipping 4 (already done)
Skipping 5 (already done)
Skipping 6 (already done)
Skipping 7 (already done)
Skipping 8 (already done)
Skipping 9 (already done)
Skipping 10 (already done)
Skipping 11 (already done)
Skipping 12 (already done)
Skipping 13 (already done)
Skipping 14 (already done)
Skipping 15 (already done)
Skipping 16 (already done)
Skipping 17 (already done)
Skipping 18 (already done)
Skipping 19 (already done)
Skipping 20 (already done)
Skipping 21 (already done)
Skipping 22 (already done)
Skipping 23 (already done)
Skipping 24 (already done)
Skipping 25 (already done)
Skipping 26 (already done)
Skipping 27 (already done)
Skipping 28 (already done)
Skipping 29 (already done)
Skipping 30 (already done)
Skipping 31 (already done)
Skipping 32 (already done)
Skipping 33 (already done)
Skipping 34 (already done)
Skipping 35 (already done)
Skippi

In [13]:
# ans = call_llm("answer with jokes", "what is the capital of  poland")
# print(ans)

In [11]:
# print("problem ang", task['problem'])
# print("problem polish", problem)

# print("solution ang", task['solution'])
# print("solution polish", solution)